In [ ]:
!pip -q install pydataxm pandas requests

In [ ]:
from pydataxm.pydatasimem import CatalogSIMEM

# Consultar el catálogo de conjuntos de datos de SIMEM
catalogo = CatalogSIMEM("Datasets")
datos = catalogo.get_data()

# Buscar conjuntos relacionados con la demanda real
nombre_columna = next(
    c for c in datos.columns
    if "nombreconjuntodatos" in c.lower()
)

coincidencias = datos[
    datos[nombre_columna]
    .astype(str)
    .str.contains("demanda real nacional", case=False, na=False)
]

display(coincidencias)

,idDataset,nombreConjuntoDatos,fechaPublicacion,fechaActualizacion,inicioDato,finDato,fechaDescarga,urlConexionAPI,urlConjuntoDatos,tipoPublicacion
157,14fabb,Demanda real nacional,2025-10-14T16:22:33.677,2026-10-19T04:32:20.19,2021-01-01T00:00:00,2026-10-14T00:00:00,None,https://www.simem.coPublicData?startDate=2026-...,https://www.simem.co/datadetail/14fabb20-e4ec-...,Público


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import os

carpeta = "/content/drive/MyDrive/Tesis_XM"
os.makedirs(carpeta, exist_ok=True)

print("Carpeta creada:", carpeta)

Mounted at /content/drive
Carpeta creada: /content/drive/MyDrive/Tesis_XM


In [ ]:
import pandas as pd
import os
import time
from pydataxm.pydatasimem import ReadSIMEM

# CONFIGURACIÓN
dataset_id = "14fabb"  # Verifica el ID antes de descargar
fecha_inicio = "2015-01-01"
fecha_fin = "2026-10-19"

archivo = os.path.join(
    carpeta,
    "demanda_real_nacional.csv"
)

# Crear períodos mensuales
meses = pd.date_range(
    start=fecha_inicio,
    end=fecha_fin,
    freq="MS"
)

for i, inicio in enumerate(meses):

    fin_mes = inicio + pd.offsets.MonthEnd(0)
    fin_mes = min(fin_mes, pd.Timestamp(fecha_fin))

    inicio_txt = inicio.strftime("%Y-%m-%d")
    fin_txt = fin_mes.strftime("%Y-%m-%d")

    print(f"Descargando {inicio_txt} a {fin_txt}")

    try:
        consulta = ReadSIMEM(
            dataset_id,
            inicio_txt,
            fin_txt
        )

        df_mes = consulta.main(filter=False)

        if df_mes is not None and not df_mes.empty:

            # Guardar inmediatamente cada mes
            df_mes.to_csv(
                archivo,
                mode="w" if i == 0 else "a",
                header=(i == 0),
                index=False,
                encoding="utf-8-sig"
            )

            print("Registros descargados:", len(df_mes))

        else:
            print("Sin registros para este período.")

    except Exception as e:
        print(f"Error en {inicio_txt}: {e}")
        print("Anota este mes para volver a consultarlo.")
        break

    # Evitar solicitudes demasiado seguidas
    time.sleep(1)

print("Proceso terminado. Archivo:", archivo)

Descargando 2015-01-01 a 2015-01-31
****************************************************************************************************
Initializing object
The object has been initialized with the dataset: "Demanda real nacional"
****************************************************************************************************
Inicio consulta sincronica
Creacion url: 0.0005106925964355469
For the URL: https://www.simem.co/backend-files/api/datos-publicos?startDate=2015-01-01&endDate=2015-01-31&datasetId=14fabb
There are 0 records
Extraccion de registros: 1.9451026916503906
End of data extracting process
****************************************************************************************************
Sin registros para este período.
Descargando 2015-02-01 a 2015-02-28
****************************************************************************************************
Initializing object
The object has been initialized with the dataset: "Demanda real nacional"
*********************

In [ ]:
import pandas as pd

df = pd.read_csv(archivo)

print("Total de registros:", len(df))
print("Columnas:", df.columns.tolist())
display(df.head())

# Revisar las fechas disponibles
for columna in df.columns:
    if "fecha" in columna.lower():
        fechas = pd.to_datetime(df[columna], errors="coerce")
        print("Columna de fecha:", columna)
        print("Fecha inicial:", fechas.min())
        print("Fecha final:", fechas.max())

Total de registros: 7902119
Columnas: ['DdaReal', '2021-01-24T23:00:00', 'RQEC', 'No Regulado', 'TX4', '5.7', 'kWh', 'PT1H']


,DdaReal,2021-01-24T23:00:00,RQEC,No Regulado,TX4,5.7,kWh,PT1H
0,DdaReal,2021-01-24T22:00:00,RQEC,No Regulado,TX4,5.63,kWh,PT1H
1,DdaReal,2021-01-24T21:00:00,RQEC,No Regulado,TX4,5.49,kWh,PT1H
2,DdaReal,2021-01-24T20:00:00,RQEC,No Regulado,TX4,5.78,kWh,PT1H
3,DdaReal,2021-01-24T19:00:00,RQEC,No Regulado,TX4,5.85,kWh,PT1H
4,DdaReal,2021-01-24T18:00:00,RQEC,No Regulado,TX4,5.78,kWh,PT1H
